# Running Open WebUI in Google Colab



### Introduction: Your Local Open WebUI Workspace in the Cloud

Welcome to your Open WebUI workspace! In this environment, we have configured a completely private, single-user instance of Open WebUI running inside a local `tmux` session. Because this notebook is fully self-contained and isolated, we have streamlined the experience for you: all automatic internet downloads are disabled, and user authentication has been bypassed. This means the server will boot instantly, and you won't have to create an admin account or log in. Simply launch the workspace, and you will be dropped directly into the chat interface where you can immediately interact with your locally hosted large language models ( LLMs ) and begin your data science workflows.




### References

- [Build Your Own AI Lab]( https://learning.oreilly.com/course/build-your-own/9780135439616/ )





## Base setup



In [ ]:
%%capture
%%bash
pip install ollama


In [ ]:
import requests
from time import sleep
import ollama
import textwrap
from datetime import datetime, timezone
from google.colab import output


## Setup Open WebUI


### Install Open WebUI


In [ ]:
%%writefile /tmp/tmux.open-webui.install.sh
#!/bin/bash

# open-webui install
mkdir -p /tmp/open-webui-logs/
exec > >(tee /tmp/open-webui-logs/open-webui.install.log) 2>&1
rm -f /tmp/.done.open-webui

echo == create virtual env
cd /content/
uv init webui --python 3.11
cd webui/

echo == activate virtual env
uv sync

echo == open-webui install
uv add open-webui
touch /tmp/.done.open-webui


Writing /tmp/tmux.open-webui.install.sh


In [ ]:
%%bash

# Start and run the install script
tmux new-session -s open-webui-install -d 'bash /tmp/tmux.open-webui.install.sh'


### Launch Open WebUI


In [ ]:
%%writefile /tmp/tmux.webui.launch.sh
#!/bin/bash

# open-webui install
mkdir -p /tmp/open-webui-logs/
exec > >(tee /tmp/open-webui-logs/open-webui.launch.log) 2>&1

# wait for setup to finish
until [ -f /tmp/.done.open-webui ] ; do
  date
  sleep 1
done

cd /content/webui/

echo == open-webui start
# Bypass Login Screen entirely
export WEBUI_AUTH="False"

# Bypass Hugging Face Downloads & Network checks
export OFFLINE_MODE=True
export RAG_EMBEDDING_MODEL_AUTO_UPDATE=False
export RAG_RERANKING_MODEL_AUTO_UPDATE=False
export WHISPER_MODEL_AUTO_UPDATE=False

# Launch
uv run open-webui serve --host 127.0.0.1 --port 9080

echo == Done
sleep 1000



Writing /tmp/tmux.webui.launch.sh


In [ ]:
%%bash

# Start launch session
tmux new-session -s open-webui-launch -d

# Run the launch script
tmux send-keys -t open-webui-launch 'bash /tmp/tmux.webui.launch.sh' Enter


## Setup Ollama



### Install Ollama


In [ ]:
%%writefile /tmp/tmux.ollama.setup.sh
#!/bin/bash

# ollama service install and launch
rm -f /tmp/.done.ollama
mkdir -p /tmp/ollama-logs/
exec > >(tee /tmp/ollama-logs/ollama.setup.log) 2>&1
until which zstd ; do
  apt-get update
  apt-get install -y zstd
done

curl -fsSL https://ollama.com/install.sh | sh
touch /tmp/.done.ollama


Writing /tmp/tmux.ollama.setup.sh


In [ ]:
%%bash

# Start and run the setup script
tmux new -s ollama-setup -d 'bash /tmp/tmux.ollama.setup.sh'


### Launch Ollama


In [ ]:
%%writefile /tmp/tmux.ollama.launch.sh
#!/bin/bash

mkdir -p /tmp/ollama-logs/
exec > >(tee /tmp/ollama-logs/ollama.launch.log) 2>&1

# wait for setup to finish
until [ -f /tmp/.done.ollama ] ; do
  date
  sleep 1
done

# OLLAMA_CUDA=1
OLLAMA_KEEP_ALIVE=20m ollama serve
echo == Done
sleep 10


Writing /tmp/tmux.ollama.launch.sh


In [ ]:
%%bash

# Start launch session
tmux new-session -s ollama-launch -d

# Run the launch script
tmux send-keys -t ollama-launch 'bash /tmp/tmux.ollama.launch.sh' Enter


### Get a model


In [ ]:
%%writefile /tmp/tmux.ollama.model.sh
#!/bin/bash

mkdir -p /tmp/ollama-logs/
exec > >(tee /tmp/ollama-logs/ollama.model.log) 2>&1

# wait for ollama service to start
until curl -s -I http://localhost:11434/api/tags ; do
  date
  sleep 1
done

# pull a small model and load into VRAM
ollama pull llama3.2
ollama run llama3.2:latest ''


Writing /tmp/tmux.ollama.model.sh


In [ ]:
%%bash

# Start and run the model script
tmux new-session -s ollama-model -d 'bash /tmp/tmux.ollama.model.sh'


## Use Open WebUI


In [ ]:
url = "http://127.0.0.1:9080"

print("Waiting for WebUI to start")
for i in range(1,301):
  try:
    requests.head( url )
    print()
    break
  except:
    print("=", end="")
    if i % 30 == 0 :
      print(f" -- {i:3}s")
    sleep(1)
print(f"{i} seconds")

print("Open WebUI has started")
output.serve_kernel_port_as_window(9080)


Waiting for WebUI to start

1 seconds
Open WebUI has started
Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>

## Use Ollama via the terminal


In [ ]:
!ollama list


NAME               ID              SIZE      MODIFIED           
llama3.2:latest    a80c4f17acd5    2.0 GB    About a minute ago    


In [ ]:
!ollama run llama3.2:latest ''


In [ ]:
!ollama ps


NAME               ID              SIZE      PROCESSOR    CONTEXT    UNTIL               
llama3.2:latest    a80c4f17acd5    2.6 GB    100% GPU     4096       18 minutes from now    


In [ ]:
!echo "In one sentence, what is CS50?"


In one sentence, what is CS50?
